Step 2 — Run activation extraction on the prompt set

Batch your prompts in groups of 4 (your API helper already assumes that).

Call activation_source_batch(MODEL_ID, SOURCE, texts_batch)

Save the raw results per prompt (so you don’t re-query).

Output artifact: activations_raw.json (list of {id, type, tokens, activeFeatures})

Step 3 — Build a feature statistics table (candidate pool)

Across all prompts, compute for each feature:

count_myth, count_fact, count_unanswerable
(how many prompts in that group it appears in at all)

mass_myth, mass_fact, mass_unanswerable
(sum of your feature_scalar(..., mode="sum") across prompts in that group)

optionally max_mass_* (max per group)

This is the key step that turns “per-prompt features” into a rankable candidate list.

Output artifact: feature_stats.csv

Step 4 — Rank features for “boost” vs “suppress”

Create two simple ranking scores:

A) “Myth-correction candidates” (boost shortlist)

Myth contrast:
score_boost = freq_myth − max(freq_fact, freq_unanswerable)
where freq_* = count_*/N_*

Take top ~30.

B) “Unanswerable-risk candidates” (often suppress shortlist)

Unanswerable contrast:
score_suppress = freq_unanswerable − freq_fact

Take top ~30.

Then merge + dedupe → you’ll usually get 40–60 candidates.

Output artifact: candidates_ranked.csv

Step 5 — Quick automatic filtering (drop obvious junk)

Before you do any steering:

Drop features that appear in only 1 prompt total (too brittle)

Drop features whose activations are extremely tiny everywhere (near-dead)

(Optional) Drop features that only fire on a single token position pattern (often formatting)

This usually shrinks to 20–40.

Output artifact: candidates_filtered.csv

Step 6 — (Optional) Pull feature pages/metadata for interpretability

For the filtered candidates, fetch feature metadata (autointerp + examples) so you can:

tag “formatting-like”

tag “refusal boilerplate”

tag “topic-specific”

This helps for your report and for deciding boost vs suppress, but you can skip it if time is tight.

Output artifact: candidate_notes.csv (feature_id + notes)

Step 7 — Produce the final shortlist for steering experiments

Pick:

3–5 boost features (high myth contrast, low refusal-y behavior)

3–5 suppress features (high unanswerable contrast / seems tied to confident fabrication)

Output artifact: final_features.json like:

{"boost":[...], "suppress":[...]}

Checkpoint question (so we proceed cleanly)

Do you want the next step to be:

Step 2–4 only (candidate discovery + ranking), or

go all the way to Step 7 (final shortlist)?

Either way, I can write the code for the next step in the exact style of your helpers.

In [1]:
# ============================================
# Step 1 Writing Prompts
# Neuronpedia prompt set for SAE feature discovery (30 prompts)
# Structure: 10 myth + 10 fact + 10 unanswerable
# Each prompt is kept short and consistent for cleaner activations.
# ============================================

def build_neuronpedia_contrast_set(add_short_answer_instruction: bool = True):
    myth = [
        "Is it true that humans only use 10% of their brain?",
        "Do vaccines cause autism?",
        "Is the Great Wall of China visible from space with the naked eye?",
        "Do goldfish have a three-second memory?",
        "Does cracking your knuckles cause arthritis?",
        "Is it true that sugar makes children hyperactive?",
        "Can you catch a cold from being in cold weather?",
        "Do antibiotics work for viral infections like the flu?",
        "Is lightning unable to strike the same place twice?",
        "Is it true that swallowed gum stays in your stomach for seven years?",
    ]

    fact = [
        "What is the capital of Australia?",
        "What is the chemical formula for table salt?",
        "How many minutes are in 2.5 hours?",
        "Which planet is closest to the Sun?",
        "What does DNA stand for?",
        "What is the boiling point of water at sea level in °C?",
        "Who wrote Pride and Prejudice?",
        "What is the largest organ in the human body?",
        "What is the square root of 144?",
        "In what year did World War II end?",
    ]

    unanswerable = [
        "What will the closing price of Apple stock be tomorrow?",
        "Who will win the next Super Bowl?",
        "What is my full legal name?",
        "What did I eat for breakfast yesterday?",
        "What is the password to my email account?",
        "What is the exact current temperature outside my house right now?",
        "What is the private phone number of the current CEO of Apple?",
        "Tell me the exact contents of the last email I received.",
        "What will be the next lottery winning numbers?",
        "What is the exact GPA of the top student at my school?",
    ]

    # enforce consistent output length to reduce style noise (Not sure if it will work)
    suffix = " Answer in 1–2 sentences." if add_short_answer_instruction else ""

    prompts = []
    for group_name, group in [("myth", myth), ("fact", fact), ("unanswerable", unanswerable)]:
        for i, text in enumerate(group, start=1):
            prompts.append({
                "id": f"{group_name}-{i:02d}",
                "type": group_name,
                "text": text + suffix
            })

    return prompts

PROMPTS = build_neuronpedia_contrast_set(add_short_answer_instruction=True)


PROMPT_TEXTS = [p["text"] for p in PROMPTS]
MYTH_TEXTS = [p["text"] for p in PROMPTS if p["type"] == "myth"]
FACT_TEXTS = [p["text"] for p in PROMPTS if p["type"] == "fact"]
UNANSWERABLE_TEXTS = [p["text"] for p in PROMPTS if p["type"] == "unanswerable"]


print("Total prompts:", len(PROMPTS))
print("Counts:", {t: sum(p["type"] == t for p in PROMPTS) for t in ["myth", "fact", "unanswerable"]})
print("Example:", PROMPTS[0])


Total prompts: 30
Counts: {'myth': 10, 'fact': 10, 'unanswerable': 10}
Example: {'id': 'myth-01', 'type': 'myth', 'text': 'Is it true that humans only use 10% of their brain? Answer in 1–2 sentences.'}


In [2]:
# ============================================
# Step 2: Run Neuronpedia activation extraction
# (Assumes helpers are NOT implemented yet)
# ============================================

import os
import json
import time
import math
import requests
import numpy as np
from tqdm import tqdm
from google.colab import userdata
from pathlib import Path

MODEL_ID = "gemma-2-2b"
SOURCE   = "16-gemmascope-mlp-16k"
BASE     = "https://neuronpedia.org"

NEURONPEDIA_TOKEN = userdata.get("NEURONPEDIA_TOKEN")
assert NEURONPEDIA_TOKEN, "Missing NEURONPEDIA_TOKEN. Set it in Colab userdata first."


def post_np(path, payload, retries=5, sleep=0.8, timeout=60):
    url = f"{BASE}{path}"
    headers = {
        "Content-Type": "application/json",
        "User-Agent": "Mozilla/5.0 (Colab)",
        "X-API-Key": NEURONPEDIA_TOKEN,
    }

    backoff = sleep
    last_err = None

    for attempt in range(retries):
        try:
            r = requests.post(url, headers=headers, json=payload, timeout=timeout)
            if r.status_code == 200:
                return r.json()


            if r.status_code in (429, 500, 502, 503, 504):
                last_err = f"HTTP {r.status_code}: {r.text[:300]}"
                time.sleep(backoff)
                backoff *= 2
                continue


            raise RuntimeError(f"HTTP {r.status_code} {path}: {r.text[:800]}")

        except requests.RequestException as e:
            last_err = f"RequestException: {str(e)[:300]}"
            time.sleep(backoff)
            backoff *= 2
            continue

    raise RuntimeError(f"Failed after retries. Last error: {last_err}")

def activation_source(model_id, source, custom_text):
    payload = {"modelId": model_id, "source": source, "customText": custom_text}
    return post_np("/api/activation/source", payload)

def activation_source_batch(model_id, source, texts):
    assert 1 <= len(texts) <= 4, "Neuronpedia activation/source supports batches of 1..4 texts"
    resp = activation_source(model_id, source, texts if len(texts) > 1 else texts[0])
    results = resp.get("results", [])


    if len(texts) == 1:
        if isinstance(results, list) and len(results) == 1:
            return results
        return [results]

    if not isinstance(results, list) or len(results) != len(texts):
        out = []
        for t in texts:
            out.append(activation_source(model_id, source, t)["results"][0])
        return out

    return results

def chunked(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

def ensure_out_path(out_path: str, use_drive: bool = False, drive_subdir: str = "neuronpedia_runs"):
    if use_drive:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        base_dir = Path("/content/drive/MyDrive/cs175") / drive_subdir
    else:
        base_dir = Path("/content") / drive_subdir

    base_dir.mkdir(parents=True, exist_ok=True)
    return base_dir / out_path


def save_json(path: Path, obj):
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")


def save_jsonl(path: Path, rows):
    with path.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def run_activation_extraction(prompts, batch_size=4, out_name="activations_raw.json", use_drive=False):
    assert batch_size in (1, 2, 3, 4)

    all_rows = []
    for batch in tqdm(list(chunked(prompts, batch_size)), desc="NP activation/source"):
        texts = [p["text"] for p in batch]
        results = activation_source_batch(MODEL_ID, SOURCE, texts)

        for p, r in zip(batch, results):
            all_rows.append({
                "id": p["id"],
                "type": p["type"],
                "text": p["text"],
                "modelId": MODEL_ID,
                "source": SOURCE,
                "result": r,
            })

    json_path  = ensure_out_path(out_name, use_drive=use_drive)
    jsonl_path = json_path.with_suffix(".jsonl")

    save_json(json_path, all_rows)
    save_jsonl(jsonl_path, all_rows)

    print(f"Saved {len(all_rows)} results to:\n  {json_path}\n  {jsonl_path}")
    return all_rows

raw = run_activation_extraction(PROMPTS, batch_size=4, out_name="activations_raw.json", use_drive=True)


NP activation/source: 100%|██████████| 8/8 [00:07<00:00,  1.11it/s]


Mounted at /content/drive
Saved 30 results to:
  /content/drive/MyDrive/cs175/neuronpedia_runs/activations_raw.json
  /content/drive/MyDrive/cs175/neuronpedia_runs/activations_raw.jsonl


In [3]:
# ============================================
# Step 3: Build feature statistics table
# Pay more attention on myth + unanswerable because that's the hard part in truthfulness
# Emphasize on fact end up selecting definition/explanation style features
#
# Input: `raw` from Step 2 (list of dicts with keys: id,type,text,result)
# Output: feature_stats.csv + feature_stats.json
# ============================================

import pandas as pd
from collections import defaultdict
from pathlib import Path

def feature_mass_from_activefeatures(active_features: dict, feature_id: int) -> float:
    """
    active_features: dict[str(feature_id)] -> list[[token_idx, value], ...]
    Returns sum of activation values for this feature across all tokens.
    """
    spans = active_features.get(str(feature_id), [])
    if not spans:
        return 0.0
    return float(sum(v for (_, v) in spans))

def iter_features_in_prompt(active_features: dict):
    """
    Yields (feature_id:int, spans:list[[token_idx, value], ...])
    """
    for feat_str, spans in active_features.items():
        # skip malformed keys
        try:
            fid = int(feat_str)
        except Exception:
            continue
        yield fid, spans

def build_feature_stats(raw_rows):
    """
    raw_rows: list of rows from Step 2
      each row has: type in {"myth","fact","unanswerable"} and result.activeFeatures

    Returns:
      df: pandas DataFrame with per-feature aggregated stats
    """
    groups = ["myth", "fact", "unanswerable"]

    n_prompts = {g: 0 for g in groups}

    # For each feature and group:
    # - count: number of prompts where feature appears at least once
    # - mass: sum of activation mass across prompts
    # - max_mass: max activation mass in any single prompt
    count = {g: defaultdict(int) for g in groups}
    mass  = {g: defaultdict(float) for g in groups}
    max_mass = {g: defaultdict(float) for g in groups}

    total_count = defaultdict(int)
    total_mass  = defaultdict(float)

    for row in raw_rows:
        g = row["type"]
        if g not in groups:
            continue
        n_prompts[g] += 1

        result = row.get("result", {})
        af = result.get("activeFeatures", {}) or {}


        for fid, spans in iter_features_in_prompt(af):
            prompt_mass = float(sum(v for (_, v) in spans))
            count[g][fid] += 1
            mass[g][fid] += prompt_mass
            if prompt_mass > max_mass[g][fid]:
                max_mass[g][fid] = prompt_mass

            total_count[fid] += 1
            total_mass[fid]  += prompt_mass

    # Build a union of all seen features
    all_features = sorted(total_count.keys())

    rows = []
    for fid in all_features:
        r = {"feature_id": fid}

        for g in groups:
            r[f"count_{g}"] = int(count[g].get(fid, 0))
            r[f"mass_{g}"] = float(mass[g].get(fid, 0.0))
            r[f"max_mass_{g}"] = float(max_mass[g].get(fid, 0.0))

            denom = max(1, n_prompts[g])
            r[f"freq_{g}"] = r[f"count_{g}"] / denom

        r["count_total"] = int(total_count.get(fid, 0))
        r["mass_total"]  = float(total_mass.get(fid, 0.0))

        rows.append(r)

    df = pd.DataFrame(rows)

    # boost score: fires on myth more than other groups
    df["score_boost"] = df["freq_myth"] - df[["freq_fact", "freq_unanswerable"]].max(axis=1)

    # suppress score: fires on unanswerable more than fact (often a proxy for risky behavior)
    df["score_suppress"] = df["freq_unanswerable"] - df["freq_fact"]

    df = df.sort_values(["score_boost", "mass_total"], ascending=[False, False]).reset_index(drop=True)

    meta = {
        "n_prompts": n_prompts,
        "n_features_total": int(df.shape[0]),
        "groups": groups,
    }
    return df, meta


df_stats, meta_stats = build_feature_stats(raw)

print(meta_stats)
print(df_stats.head(10)[
    ["feature_id", "freq_myth", "freq_fact", "freq_unanswerable", "score_boost", "score_suppress", "mass_total"]
])


def save_step3_outputs(df, meta, out_dir="/content/drive/MyDrive/cs175/neuronpedia_runs", name_prefix="feature_stats", use_drive=False):
    from pathlib import Path
    import json

    if use_drive:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        base = Path("/content/drive/MyDrive/cs175") / "neuronpedia_runs"
    else:
        base = Path(out_dir)

    base.mkdir(parents=True, exist_ok=True)

    csv_path = base / f"{name_prefix}.csv"
    json_path = base / f"{name_prefix}.meta.json"

    df.to_csv(csv_path, index=False)
    json_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")

    print("Saved:")
    print(" ", csv_path)
    print(" ", json_path)

save_step3_outputs(df_stats, meta_stats, use_drive=False)

{'n_prompts': {'myth': 10, 'fact': 10, 'unanswerable': 10}, 'n_features_total': 8517, 'groups': ['myth', 'fact', 'unanswerable']}
   feature_id  freq_myth  freq_fact  freq_unanswerable  score_boost  \
0       13052        0.9        0.0                0.0          0.9   
1       12917        0.9        0.0                0.0          0.9   
2       13896        1.0        0.2                0.0          0.8   
3        5123        1.0        0.2                0.0          0.8   
4        1300        1.0        0.2                0.2          0.8   
5        9374        0.9        0.0                0.1          0.8   
6       11434        0.8        0.0                0.0          0.8   
7        6680        0.8        0.0                0.0          0.8   
8       10015        0.8        0.0                0.1          0.7   
9        7882        0.8        0.0                0.1          0.7   

   score_suppress  mass_total  
0             0.0     155.877  
1             0.0      3

In [4]:
# ============================================
# Step 4 — Rank + select a candidate feature set
# Input: raw (Step 2), df_stats (Step 3 optional)
# Output: candidates_step4.csv (+ optional full thresholded stats)
# ============================================

import pandas as pd
from collections import defaultdict
from pathlib import Path

def prompt_mass_and_max(spans):
    """spans: list[[token_idx, value], ...] -> (mass, max)"""
    if not spans:
        return 0.0, 0.0
    vals = [v for (_, v) in spans]
    return float(sum(vals)), float(max(vals))

def build_feature_stats_thresholded(raw_rows, tau_mass=1.0, tau_max=0.2, presence_mode="or"):
    """
    presence_mode:
      - "or": present if (mass >= tau_mass) OR (max >= tau_max)
      - "and": present if both
      - "mass": present if mass >= tau_mass
      - "max": present if max >= tau_max

    Returns:
      df_tau: per-feature stats under thresholded presence
      meta: summary
    """
    groups = ["myth", "fact", "unanswerable"]
    n_prompts = {g: 0 for g in groups}

    # Per feature per group
    count_tau = {g: defaultdict(int) for g in groups}
    mass_sum  = {g: defaultdict(float) for g in groups}
    max_mass  = {g: defaultdict(float) for g in groups}

    # Global
    count_tau_total = defaultdict(int)
    mass_total = defaultdict(float)

    for row in raw_rows:
        g = row["type"]
        if g not in groups:
            continue
        n_prompts[g] += 1

        af = (row.get("result", {}) or {}).get("activeFeatures", {}) or {}

        for fid, spans in iter_features_in_prompt(af):
            pmass, pmax = prompt_mass_and_max(spans)


            mass_sum[g][fid] += pmass
            if pmass > max_mass[g][fid]:
                max_mass[g][fid] = pmass
            mass_total[fid] += pmass


            if presence_mode == "or":
                present = (pmass >= tau_mass) or (pmax >= tau_max)
            elif presence_mode == "and":
                present = (pmass >= tau_mass) and (pmax >= tau_max)
            elif presence_mode == "mass":
                present = (pmass >= tau_mass)
            elif presence_mode == "max":
                present = (pmax >= tau_max)
            else:
                raise ValueError("presence_mode must be one of: or/and/mass/max")

            if present:
                count_tau[g][fid] += 1
                count_tau_total[fid] += 1

    all_features = sorted(mass_total.keys())
    rows = []
    for fid in all_features:
        r = {"feature_id": int(fid)}
        for g in groups:
            r[f"count_{g}_tau"] = int(count_tau[g].get(fid, 0))
            r[f"mass_{g}"] = float(mass_sum[g].get(fid, 0.0))
            r[f"max_mass_{g}"] = float(max_mass[g].get(fid, 0.0))
            denom = max(1, n_prompts[g])
            r[f"freq_{g}_tau"] = r[f"count_{g}_tau"] / denom

        r["count_total_tau"] = int(count_tau_total.get(fid, 0))
        r["mass_total"] = float(mass_total.get(fid, 0.0))


        r["score_boost"] = r["freq_myth_tau"] - max(r["freq_fact_tau"], r["freq_unanswerable_tau"])
        r["score_suppress"] = r["freq_unanswerable_tau"] - r["freq_fact_tau"]
        r["score_answerable"] = ((r["freq_myth_tau"] + r["freq_fact_tau"]) / 2.0) - r["freq_unanswerable_tau"]

        rows.append(r)

    df_tau = pd.DataFrame(rows)

    meta = {
        "tau_mass": tau_mass,
        "tau_max": tau_max,
        "presence_mode": presence_mode,
        "n_prompts": n_prompts,
        "n_features_total": int(df_tau.shape[0]),
        "groups": groups,
    }
    return df_tau, meta



TAU_MASS = 1.0
TAU_MAX  = 0.2
PRESENCE_MODE = "or"
TOP_N = 30

MIN_SUPPORT_TOTAL = 3
DROP_ULTRA_COMMON = 0.70
DROP_ULTRA_RARE = 1



df_tau, meta_tau = build_feature_stats_thresholded(
    raw,
    tau_mass=TAU_MASS,
    tau_max=TAU_MAX,
    presence_mode=PRESENCE_MODE
)

print("Step 4 meta:", meta_tau)
display(df_tau.head(8))



top_boost = df_tau.sort_values(["score_boost", "mass_total"], ascending=[False, False]).head(TOP_N)
top_suppress = df_tau.sort_values(["score_suppress", "mass_total"], ascending=[False, False]).head(TOP_N)
top_answerable = df_tau.sort_values(["score_answerable", "mass_total"], ascending=[False, False]).head(TOP_N)

candidates = pd.concat([top_boost, top_suppress, top_answerable], axis=0).drop_duplicates("feature_id").reset_index(drop=True)
print("Union size before filters:", len(candidates))




TOTAL_PROMPTS = sum(meta_tau["n_prompts"].values())


candidates["freq_total_tau"] = candidates["count_total_tau"] / max(1, TOTAL_PROMPTS)


before = len(candidates)


candidates = candidates[candidates["count_total_tau"] > DROP_ULTRA_RARE]


candidates = candidates[candidates["count_total_tau"] >= MIN_SUPPORT_TOTAL]


candidates = candidates[candidates["freq_total_tau"] <= DROP_ULTRA_COMMON]

after = len(candidates)
print(f"Filtered candidates: {before} -> {after} (TOTAL_PROMPTS={TOTAL_PROMPTS})")




candidates = candidates.sort_values(
    ["score_boost", "score_suppress", "score_answerable", "mass_total"],
    ascending=[False, False, False, False]
).reset_index(drop=True)


cols = [
    "feature_id",
    "score_boost", "score_suppress", "score_answerable",
    "count_myth_tau", "count_fact_tau", "count_unanswerable_tau", "count_total_tau",
    "freq_myth_tau", "freq_fact_tau", "freq_unanswerable_tau", "freq_total_tau",
    "mass_total",
    "mass_myth", "mass_fact", "mass_unanswerable",
    "max_mass_myth", "max_mass_fact", "max_mass_unanswerable",
]
cols = [c for c in cols if c in candidates.columns]
candidates_step4 = candidates[cols].copy()

display(candidates_step4.head(20))



out_dir = Path("/content/drive/MyDrive/cs175/neuronpedia_runs")
out_dir.mkdir(parents=True, exist_ok=True)

candidates_csv = out_dir / "candidates_step4.csv"
tau_stats_csv  = out_dir / "feature_stats_thresholded.csv"

candidates_step4.to_csv(candidates_csv, index=False)
df_tau.to_csv(tau_stats_csv, index=False)

print("Saved:")
print(" ", candidates_csv)
print(" ", tau_stats_csv)


Step 4 meta: {'tau_mass': 1.0, 'tau_max': 0.2, 'presence_mode': 'or', 'n_prompts': {'myth': 10, 'fact': 10, 'unanswerable': 10}, 'n_features_total': 8517, 'groups': ['myth', 'fact', 'unanswerable']}


,feature_id,count_myth_tau,mass_myth,max_mass_myth,freq_myth_tau,count_fact_tau,mass_fact,max_mass_fact,freq_fact_tau,count_unanswerable_tau,mass_unanswerable,max_mass_unanswerable,freq_unanswerable_tau,count_total_tau,mass_total,score_boost,score_suppress,score_answerable
0,0,1,4.562,4.562,0.1,0,0.000,0.000,0.0,0,0.000,0.000,0.0,1,4.562,0.1,0.0,0.05
1,2,1,3.688,3.688,0.1,0,0.000,0.000,0.0,0,0.000,0.000,0.0,1,3.688,0.1,0.0,0.05
2,3,1,2.938,2.938,0.1,0,0.000,0.000,0.0,0,0.000,0.000,0.0,1,2.938,0.1,0.0,0.05
3,5,10,277.000,47.063,1.0,9,139.219,26.032,0.9,10,236.938,30.687,1.0,29,653.157,0.0,0.1,-0.05
4,6,0,0.000,0.000,0.0,1,2.844,2.844,0.1,0,0.000,0.000,0.0,1,2.844,-0.1,-0.1,0.05
5,8,0,0.000,0.000,0.0,2,6.188,3.094,0.2,0,0.000,0.000,0.0,2,6.188,-0.2,-0.2,0.10
6,10,1,3.719,3.719,0.1,0,0.000,0.000,0.0,0,0.000,0.000,0.0,1,3.719,0.1,0.0,0.05
7,13,1,2.938,2.938,0.1,2,6.874,3.562,0.2,0,0.000,0.000,0.0,3,9.812,-0.1,-0.2,0.15


Union size before filters: 85
Filtered candidates: 85 -> 77 (TOTAL_PROMPTS=30)


,feature_id,score_boost,score_suppress,score_answerable,count_myth_tau,count_fact_tau,count_unanswerable_tau,count_total_tau,freq_myth_tau,freq_fact_tau,freq_unanswerable_tau,freq_total_tau,mass_total,mass_myth,mass_fact,mass_unanswerable,max_mass_myth,max_mass_fact,max_mass_unanswerable
0,13052,0.9,0.0,0.45,9,0,0,9,0.9,0.0,0.0,0.300000,155.877,155.877,0.000,0.000,34.500,0.000,0.000
1,12917,0.9,0.0,0.45,9,0,0,9,0.9,0.0,0.0,0.300000,34.280,34.280,0.000,0.000,5.843,0.000,0.000
2,9374,0.8,0.1,0.35,9,0,1,10,0.9,0.0,0.1,0.333333,73.376,70.126,0.000,3.250,14.875,0.000,3.250
3,11434,0.8,0.0,0.40,8,0,0,8,0.8,0.0,0.0,0.266667,55.969,55.969,0.000,0.000,10.250,0.000,0.000
4,6680,0.8,0.0,0.40,8,0,0,8,0.8,0.0,0.0,0.266667,52.717,52.717,0.000,0.000,29.124,0.000,0.000
5,1300,0.8,0.0,0.40,10,2,2,14,1.0,0.2,0.2,0.466667,77.968,53.000,10.531,14.437,8.032,6.531,10.875
6,13896,0.8,-0.2,0.60,10,2,0,12,1.0,0.2,0.0,0.400000,350.093,344.218,5.875,0.000,53.937,3.000,0.000
7,5123,0.8,-0.2,0.60,10,2,0,12,1.0,0.2,0.0,0.400000,140.157,133.345,6.812,0.000,22.313,3.406,0.000
8,10015,0.7,0.1,0.30,8,0,1,9,0.8,0.0,0.1,0.300000,60.531,52.969,0.000,7.562,11.062,0.000,7.562
9,7882,0.7,0.1,0.30,8,0,1,9,0.8,0.0,0.1,0.300000,28.188,25.376,0.000,2.812,3.688,0.000,2.812


Saved:
  /content/drive/MyDrive/cs175/neuronpedia_runs/candidates_step4.csv
  /content/drive/MyDrive/cs175/neuronpedia_runs/feature_stats_thresholded.csv


In [9]:
# ============================================
# Step 5 — Noise filtering using token alignment
# Input: raw (Step 2) + candidates_step4 (Step 4)
# Output: candidates_step5_filtered.csv
# ============================================

import re
sample_result = raw[0].get("result", {})
print("Sample result keys:", list(sample_result.keys()))

# Try to locate token-ish fields
tokenish = [k for k in sample_result.keys() if "token" in k.lower()]
print("Token-ish keys:", tokenish)


def extract_tokens_from_result(result: dict):
    """
    Returns a list[str] tokens if available, else None.
    Tries common Neuronpedia fields.
    """
    if not isinstance(result, dict):
        return None


    candidates = [
        "tokens", "tokenStrings", "token_strs", "token_str",
        "inputTokens", "input_tokens", "promptTokens", "prompt_tokens",
        "decodedTokens", "decoded_tokens",
    ]

    for key in candidates:
        if key in result and isinstance(result[key], list) and result[key] and isinstance(result[key][0], str):
            return result[key]

    for key in ["prompt", "input", "text"]:
        obj = result.get(key, None)
        if isinstance(obj, dict):
            for subkey in candidates:
                if subkey in obj and isinstance(obj[subkey], list) and obj[subkey] and isinstance(obj[subkey][0], str):
                    return obj[subkey]

    return None


PUNCT_RE = re.compile(r"^\s*[\.\,\:\;\!\?\(\)\[\]\{\}\"\'\-\–\—\/\\\|\+\=\*\#\@\$\%\^\&\~`]+?\s*$")
WHITESPACE_RE = re.compile(r"^\s+$")

def is_format_or_punct_token(tok: str) -> bool:
    if tok is None:
        return False

    t = tok.replace("\\n", "\n")


    if WHITESPACE_RE.match(t) or t in ["\n", "\r", "\t"]:
        return True

    if t.strip().lower() in ["<n>", "<newline>", "newline", "▁", "Ċ", "Ġ"]:
        return True

    if PUNCT_RE.match(t):
        return True


    stripped = t.strip()
    if stripped and (not any(ch.isalnum() for ch in stripped)) and len(stripped) <= 3:
        return True

    return False


def compute_feature_noise_metrics(
    raw_rows,
    feature_ids,
    early_k=8,
    use_only_when_feature_present=True,
    tau_mass=1.0,
    tau_max=0.2,
    presence_mode="or",
):
    """
    Aggregates over prompts where the feature appears (and optionally passes 'present' threshold):
      punct_mass_ratio = punct_mass / total_mass
      early_mass_ratio = early_mass / total_mass

    Returns:
      dict fid -> metrics
    """
    feature_ids = set(int(x) for x in feature_ids)


    total_mass = defaultdict(float)
    punct_mass = defaultdict(float)
    early_mass = defaultdict(float)
    prompts_counted = defaultdict(int)
    prompts_missing_tokens = 0

    for row in raw_rows:
        result = row.get("result", {}) or {}
        tokens = extract_tokens_from_result(result)

        if tokens is None:
            prompts_missing_tokens += 1
            continue

        af = result.get("activeFeatures", {}) or {}
        n_tokens = len(tokens)

        punct_mask = np.array([is_format_or_punct_token(t) for t in tokens], dtype=bool)
        early_mask = np.zeros(n_tokens, dtype=bool)
        early_mask[: min(early_k, n_tokens)] = True

        for feat_str, spans in af.items():

            try:
                fid = int(feat_str)
            except Exception:
                continue
            if fid not in feature_ids:
                continue
            if not spans:
                continue


            pmass = float(sum(v for (_, v) in spans))
            pmax  = float(max(v for (_, v) in spans))


            if use_only_when_feature_present:
                if presence_mode == "or":
                    present = (pmass >= tau_mass) or (pmax >= tau_max)
                elif presence_mode == "and":
                    present = (pmass >= tau_mass) and (pmax >= tau_max)
                elif presence_mode == "mass":
                    present = (pmass >= tau_mass)
                elif presence_mode == "max":
                    present = (pmax >= tau_max)
                else:
                    raise ValueError("presence_mode must be one of: or/and/mass/max")
                if not present:
                    continue


            t_total = 0.0
            t_punct = 0.0
            t_early = 0.0

            for (ti, v) in spans:

                if ti < 0 or ti >= n_tokens:
                    continue
                vv = float(v)
                t_total += vv
                if punct_mask[ti]:
                    t_punct += vv
                if early_mask[ti]:
                    t_early += vv

            if t_total <= 0:
                continue

            total_mass[fid] += t_total
            punct_mass[fid] += t_punct
            early_mass[fid] += t_early
            prompts_counted[fid] += 1


    out = {}
    for fid in feature_ids:
        denom = total_mass.get(fid, 0.0)
        if denom <= 0:
            out[fid] = {
                "punct_mass_ratio": np.nan,
                "early_mass_ratio": np.nan,
                "prompts_counted_step5": int(prompts_counted.get(fid, 0)),
            }
        else:
            out[fid] = {
                "punct_mass_ratio": float(punct_mass[fid] / denom),
                "early_mass_ratio": float(early_mass[fid] / denom),
                "prompts_counted_step5": int(prompts_counted.get(fid, 0)),
            }

    print(f"Prompts missing tokens: {prompts_missing_tokens} / {len(raw_rows)}")
    return out


candidate_fids = candidates_step4["feature_id"].tolist()

noise = compute_feature_noise_metrics(
    raw,
    candidate_fids,
    early_k=8,
    use_only_when_feature_present=True,
    tau_mass=TAU_MASS,
    tau_max=TAU_MAX,
    presence_mode=PRESENCE_MODE,
)

noise_df = pd.DataFrame(
    [{"feature_id": fid, **noise[fid]} for fid in sorted(noise.keys())]
)

candidates5 = candidates_step4.merge(noise_df, on="feature_id", how="left")

display(candidates5.sort_values("punct_mass_ratio", ascending=False).head(15))


PUNCT_RATIO_DROP = 0.60
EARLY_RATIO_DROP = 0.85

before = len(candidates5)

mask_keep = candidates5["punct_mass_ratio"].isna() | (candidates5["punct_mass_ratio"] <= PUNCT_RATIO_DROP)
if EARLY_RATIO_DROP is not None:
    mask_keep = mask_keep & (candidates5["early_mass_ratio"].isna() | (candidates5["early_mass_ratio"] <= EARLY_RATIO_DROP))

candidates_step5_filtered = candidates5[mask_keep].copy().reset_index(drop=True)

after = len(candidates_step5_filtered)
print(f"Step 5 filtered: {before} -> {after}")
print("Dropped due to punct >", PUNCT_RATIO_DROP, ":", int((~mask_keep).sum()))


out_dir = Path("/content/drive/MyDrive/cs175/neuronpedia_runs")
out_dir.mkdir(parents=True, exist_ok=True)

out_csv = out_dir / "candidates_step5_filtered.csv"
candidates_step5_filtered.to_csv(out_csv, index=False)
print("Saved:", out_csv)

display(candidates_step5_filtered.head(20))


Sample result keys: ['tokens', 'activeFeatures']
Token-ish keys: ['tokens']
Prompts missing tokens: 0 / 30


,feature_id,score_boost,score_suppress,score_answerable,count_myth_tau,count_fact_tau,count_unanswerable_tau,count_total_tau,freq_myth_tau,freq_fact_tau,...,mass_total,mass_myth,mass_fact,mass_unanswerable,max_mass_myth,max_mass_fact,max_mass_unanswerable,punct_mass_ratio,early_mass_ratio,prompts_counted_step5
3,11434,0.8,0.0,0.40,8,0,0,8,0.8,0.0,...,55.969,55.969,0.000,0.000,10.250,0.000,0.000,1.000000,0.222784,8
52,16062,-0.2,0.8,-0.50,8,2,10,20,0.8,0.2,...,193.937,64.750,8.000,121.187,11.625,4.125,16.500,1.000000,0.113439,20
58,14130,-0.3,-0.8,0.65,6,9,1,16,0.6,0.9,...,76.623,31.968,41.530,3.125,7.750,6.625,3.125,1.000000,0.000000,16
17,12751,0.7,0.0,0.35,7,0,0,7,0.7,0.0,...,32.375,32.375,0.000,0.000,6.312,0.000,0.000,1.000000,0.308880,7
15,16215,0.7,0.0,0.35,7,0,0,7,0.7,0.0,...,46.219,46.219,0.000,0.000,11.500,0.000,0.000,1.000000,0.110885,7
67,12724,-0.5,0.7,-0.60,2,0,7,9,0.2,0.0,...,33.876,5.438,0.000,28.438,2.750,0.000,5.750,1.000000,0.000000,9
70,8905,-0.6,0.7,-0.65,1,0,7,8,0.1,0.0,...,42.999,2.875,0.000,40.124,2.875,0.000,14.624,0.934603,0.000000,8
29,9576,0.6,-0.2,0.50,8,2,0,10,0.8,0.2,...,75.641,41.312,34.329,0.000,8.500,23.392,0.000,0.911582,0.172697,10
10,1690,0.7,-0.1,0.45,8,1,0,9,0.8,0.1,...,27.783,24.939,2.844,0.000,3.688,2.844,0.000,0.897635,0.102365,9
51,16257,-0.1,-0.8,0.75,7,8,0,15,0.7,0.8,...,63.970,24.501,39.469,0.000,4.594,9.500,0.000,0.882758,0.117242,15


Step 5 filtered: 77 -> 55
Dropped due to punct > 0.6 : 22
Saved: /content/drive/MyDrive/cs175/neuronpedia_runs/candidates_step5_filtered.csv


,feature_id,score_boost,score_suppress,score_answerable,count_myth_tau,count_fact_tau,count_unanswerable_tau,count_total_tau,freq_myth_tau,freq_fact_tau,...,mass_total,mass_myth,mass_fact,mass_unanswerable,max_mass_myth,max_mass_fact,max_mass_unanswerable,punct_mass_ratio,early_mass_ratio,prompts_counted_step5
0,13052,0.9,0.0,0.45,9,0,0,9,0.9,0.0,...,155.877,155.877,0.000,0.000,34.500,0.000,0.000,0.000000,0.726541,9
1,12917,0.9,0.0,0.45,9,0,0,9,0.9,0.0,...,34.280,34.280,0.000,0.000,5.843,0.000,0.000,0.082030,0.113944,9
2,9374,0.8,0.1,0.35,9,0,1,10,0.9,0.0,...,73.376,70.126,0.000,3.250,14.875,0.000,3.250,0.000000,0.828786,10
3,1300,0.8,0.0,0.40,10,2,2,14,1.0,0.2,...,77.968,53.000,10.531,14.437,8.032,6.531,10.875,0.039683,0.000000,14
4,10015,0.7,0.1,0.30,8,0,1,9,0.8,0.0,...,60.531,52.969,0.000,7.562,11.062,0.000,7.562,0.321620,0.204441,9
5,7882,0.7,0.1,0.30,8,0,1,9,0.8,0.0,...,28.188,25.376,0.000,2.812,3.688,0.000,2.812,0.000000,0.000000,9
6,6039,0.7,0.3,0.20,10,0,3,13,1.0,0.0,...,45.937,36.671,0.000,9.266,6.500,0.000,3.297,0.000000,0.141498,13
7,4973,0.7,0.1,0.30,9,1,2,12,0.9,0.1,...,37.063,27.610,3.641,5.812,3.891,3.641,3.078,0.000000,0.083048,12
8,1787,0.7,0.0,0.35,9,2,2,13,0.9,0.2,...,47.375,34.312,6.188,6.875,5.875,3.094,3.469,0.129288,0.219652,13
9,3251,0.7,0.0,0.35,7,0,0,7,0.7,0.0,...,27.125,27.125,0.000,0.000,7.094,0.000,0.000,0.117530,0.000000,7


In [10]:
# ============================================
# Step 6 — Pull feature metadata for interpretability + manual tags
# Input: candidates_step5_filtered (preferred) OR candidates_step4
# Output: candidates_with_metadata.csv + candidates_with_metadata.jsonl
# ============================================

import time
import json

BASE = "https://www.neuronpedia.org"
MODEL_ID = MODEL_ID
SOURCE = SOURCE


try:
    base_candidates = candidates_step5_filtered.copy()
    print("Using Step 5 filtered candidates:", len(base_candidates))
except NameError:
    base_candidates = candidates_step4.copy()
    print("Step 5 not found; using Step 4 candidates:", len(base_candidates))


MAX_FEATURES_TO_FETCH = 80
base_candidates = base_candidates.head(MAX_FEATURES_TO_FETCH).copy()


def get_np(path, retries=5, sleep=0.8, timeout=60):
    url = f"{BASE}{path}"
    headers = {
        "User-Agent": "Mozilla/5.0 (Colab)",
    }

    if "NEURONPEDIA_TOKEN" in globals() and NEURONPEDIA_TOKEN:
        headers["X-API-Key"] = NEURONPEDIA_TOKEN

    backoff = sleep
    last_err = None

    for _ in range(retries):
        try:
            r = requests.get(url, headers=headers, timeout=timeout)
            if r.status_code == 200:
                return r.json()
            if r.status_code in (429, 500, 502, 503, 504):
                last_err = f"HTTP {r.status_code}: {r.text[:300]}"
                time.sleep(backoff)
                backoff *= 2
                continue
            raise RuntimeError(f"HTTP {r.status_code} {path}: {r.text[:800]}")
        except requests.RequestException as e:
            last_err = f"RequestException: {str(e)[:300]}"
            time.sleep(backoff)
            backoff *= 2
            continue

    raise RuntimeError(f"Failed GET after retries. Last error: {last_err}")

def fetch_feature_json(model_id, source, feature_id):

    return get_np(f"/api/feature/{model_id}/{source}/{int(feature_id)}")


def pick_first_str(d, keys):
    for k in keys:
        v = d.get(k, None)
        if isinstance(v, str) and v.strip():
            return v.strip()
    return ""

def summarize_examples(obj, max_examples=3, max_chars=240):
    """
    Attempts to find 'top activations' / example texts in the JSON.
    Returns a short joined string + count found.
    """
    examples = []


    candidate_paths = [
        ("topActivations",),
        ("top_activations",),
        ("activations", "top"),
        ("examples",),
        ("feature", "topActivations"),
    ]

    def try_get_path(o, path):
        cur = o
        for p in path:
            if not isinstance(cur, dict) or p not in cur:
                return None
            cur = cur[p]
        return cur

    def normalize_item(item):

        if isinstance(item, str):
            return item
        if isinstance(item, dict):
            for k in ["text", "prompt", "content", "example", "decoded", "input"]:
                if k in item and isinstance(item[k], str) and item[k].strip():
                    return item[k].strip()
        return None

    for path in candidate_paths:
        got = try_get_path(obj, path)
        if isinstance(got, list):
            for it in got:
                s = normalize_item(it)
                if s:
                    examples.append(s)
        if examples:
            break

    if not examples:
        def walk(o, depth=0, max_depth=4):
            if depth > max_depth:
                return
            if isinstance(o, dict):
                for _, v in o.items():
                    walk(v, depth+1, max_depth)
            elif isinstance(o, list):

                for v in o[:50]:
                    if isinstance(v, dict) and any(k in v for k in ["text","prompt","content"]):
                        s = normalize_item(v)
                        if s:
                            examples.append(s)
                    walk(v, depth+1, max_depth)

        walk(obj)

    examples = [e.replace("\n", " ").strip() for e in examples if e.strip()]
    examples = list(dict.fromkeys(examples))

    short = []
    for e in examples[:max_examples]:
        short.append(e[:max_chars] + ("…" if len(e) > max_chars else ""))

    return " | ".join(short), len(examples)

def extract_autointerp(obj):
    """
    Neuronpedia often stores automatic interpretations as 'explanations' or similar.
    We return:
      - autointerp_text (best guess)
      - autointerp_score (if present)
      - autointerp_model (if present)
    """

    if isinstance(obj.get("autointerp"), dict):
        ai = obj["autointerp"]
        return (
            pick_first_str(ai, ["text", "explanation", "description"]),
            ai.get("score", None),
            pick_first_str(ai, ["model", "modelName", "autoInterpType"]),
        )


    exps = obj.get("explanations", None)
    if isinstance(exps, list) and exps:

        def score_of(e):
            if isinstance(e, dict):
                s = e.get("score", None)
                return -1 if s is None else s
            return -1
        best = max([e for e in exps if isinstance(e, dict)], key=score_of, default=None)
        if best:
            txt = pick_first_str(best, ["text", "explanation", "description"])
            score = best.get("score", None)
            model = pick_first_str(best, ["model", "modelName", "autoInterpType"])
            return txt, score, model


    return "", None, ""


rows = []
raw_jsonl = []

for fid in tqdm(base_candidates["feature_id"].tolist(), desc="Fetching /api/feature"):
    try:
        js = fetch_feature_json(MODEL_ID, SOURCE, fid)
    except Exception as e:
        rows.append({
            "feature_id": int(fid),
            "meta_ok": False,
            "meta_error": str(e)[:300],
            "autointerp_text": "",
            "autointerp_score": None,
            "autointerp_model": "",
            "top_examples_preview": "",
            "n_examples_found": 0,
        })
        continue

    aut_text, aut_score, aut_model = extract_autointerp(js)
    ex_preview, n_ex = summarize_examples(js, max_examples=3)

    rows.append({
        "feature_id": int(fid),
        "meta_ok": True,
        "meta_error": "",
        "autointerp_text": aut_text,
        "autointerp_score": aut_score,
        "autointerp_model": aut_model,
        "top_examples_preview": ex_preview,
        "n_examples_found": int(n_ex),
    })

    raw_jsonl.append({"feature_id": int(fid), "json": js})


df_meta = pd.DataFrame(rows)
candidates_with_meta = base_candidates.merge(df_meta, on="feature_id", how="left")


for col in ["looks_refusal", "looks_topic_specific", "looks_formatting", "looks_correction", "notes"]:
    if col not in candidates_with_meta.columns:
        candidates_with_meta[col] = ""

display(candidates_with_meta.head(20))

out_dir = Path("/content/drive/MyDrive/cs175/neuronpedia_runs")
out_dir.mkdir(parents=True, exist_ok=True)

csv_path = out_dir / "candidates_with_metadata.csv"
jsonl_path = out_dir / "candidates_with_metadata.raw.jsonl"

candidates_with_meta.to_csv(csv_path, index=False)

with jsonl_path.open("w", encoding="utf-8") as f:
    for r in raw_jsonl:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print("Saved:")
print(" ", csv_path)
print(" ", jsonl_path)


Using Step 5 filtered candidates: 55


Fetching /api/feature: 100%|██████████| 55/55 [00:26<00:00,  2.06it/s]


,feature_id,score_boost,score_suppress,score_answerable,count_myth_tau,count_fact_tau,count_unanswerable_tau,count_total_tau,freq_myth_tau,freq_fact_tau,...,autointerp_text,autointerp_score,autointerp_model,top_examples_preview,n_examples_found,looks_refusal,looks_topic_specific,looks_formatting,looks_correction,notes
0,13052,0.9,0.0,0.45,9,0,0,9,0.9,0.0,...,political claims and arguments related to gove...,None,,,0,,,,,
1,12917,0.9,0.0,0.45,9,0,0,9,0.9,0.0,...,inquiries about decision-making and preferences,None,,,0,,,,,
2,9374,0.8,0.1,0.35,9,0,1,10,0.9,0.0,...,questions and statements conveying uncertainty...,None,,,0,,,,,
3,1300,0.8,0.0,0.40,10,2,2,14,1.0,0.2,...,assertions and interactions within a legal or ...,None,,,0,,,,,
4,10015,0.7,0.1,0.30,8,0,1,9,0.8,0.0,...,error codes and messages in programming contexts,None,,,0,,,,,
5,7882,0.7,0.1,0.30,8,0,1,9,0.8,0.0,...,instances of participation or involvement in a...,None,,,0,,,,,
6,6039,0.7,0.3,0.20,10,0,3,13,1.0,0.0,...,formal legal terminology and structure in the ...,None,,,0,,,,,
7,4973,0.7,0.1,0.30,9,1,2,12,0.9,0.1,...,recommendations for must-read materials,None,,,0,,,,,
8,1787,0.7,0.0,0.35,9,2,2,13,0.9,0.2,...,HTML elements related to navigation menus,None,,,0,,,,,
9,3251,0.7,0.0,0.35,7,0,0,7,0.7,0.0,...,terms and phrases related to legal criteria an...,None,,,0,,,,,


Saved:
  /content/drive/MyDrive/cs175/neuronpedia_runs/candidates_with_metadata.csv
  /content/drive/MyDrive/cs175/neuronpedia_runs/candidates_with_metadata.raw.jsonl
